# Lab 05｜压缩以后，任务状态和原始依据各留下什么？

这是专栏《自进化 Agent 实战》05 讲《上下文压缩：如何压缩历史才不影响任务延续？》的配套练习。练习使用构造的长任务调查材料，目标是掌握处理真实场景的能力。

## 题面

一次支付故障调查已经持续几个小时，日志和工具结果快装不进下一次模型请求了。助手目前只能说“应用发布暂时不能确认为直接根因”，还准备核对连接池，因为连接等待与错误率同时升高。有人建议删掉旧消息，只留下“发布已排除，继续查数据库”。你会怎样缩短后续模型输入，让调查继续，又不偷偷改变已有判断？原始日志退出输入以后，需要精确引用时怎么处理？请说明压缩在什么时机完成、旧历史存在哪里、你会做哪些验证。

## 参考答案

这里最容易出错的，是把“暂时不能认定”缩成“已经排除”。后者悄悄关闭了原来仍可验证的方向；只留下“查数据库”，又丢掉了选择这一步的依据。删旧消息或让模型写一句简短总结，都不能单凭输入变短就说明任务能接着做。要保留的是目标、关键事实、判断的确定程度、未决问题，以及下一步背后的理由，同时留一条“原始材料去哪里查”的线索。

压缩应当发生在原来那条任务记录上，不另起一条没有背景的新记录。发起之后要等对应的完成事件出现，不能把下一次请求当作已经压缩过的输入。它改变的是后续调用继续携带的工作集，不是把历史从系统里删掉。

这里要分清两件事：已经发生过的完整过程，和后续模型调用实际看到的工作集。系统可以继续保留较完整的历史，同时只把其中一部分交给后面的模型。压缩以后，模型不必再背着全部材料，但必须知道自己走到了哪里、依据是什么。日志逐字内容和精确数值可以继续留在文件里；摘要保留“发布前已有同类报错”及其来源，通常就够延续调查。需要精确引用时，模型若手里没有原文，就应自己再读，不能凭摘要编造；如果摘要本来就带着数值，没有回读也算正常，要同时看请求里到底有没有依据。

验证时，我会让模型复述当前任务状态：任务编号还在不在、发布是“待验证”还是被误写成“已排除”、下一步为什么是连接池、操作限制（比如不许重启）还在不在；再问两个需要精确数字的细节，看它是直接答出来还是回查文件。字符数只说明输入长度，不代表 token 或费用。


## 准备

使用 Python 3.12 内核；依赖会自动补装。需要 `GLM_API_KEY`，未设置时会弹出隐藏输入框。`lab.codex` 是真实 SDK 客户端；下面直接创建会话、运行任务、发起压缩并读取历史。`lab.watch()` 只提供超时保护，`lab.record_result()` 只保存已返回的结果。


In [ ]:
import sys
from pathlib import Path
examples = next(p for base in (Path.cwd(), *Path.cwd().parents)
                for p in (base, base / "examples") if (p / "notebook05_setup.py").is_file())
sys.path.insert(0, str(examples))
from notebook05_setup import prepare
story = prepare()


## 历史被压缩以后，调查还能接着做吗？

这里起作用的是会话内任务状态的累积：Agent 只读取得日志，再收到判断边界与操作限制的纠正。支付调查仅使用构造材料演示。
跑完对照“压缩前的依据”和“实际内容”，看任务编号、发布前报错与连接等待怎样进入同一会话。
会话由 openai_codex 的 Thread 保存；litellm 代理层把实际模型请求转给 GLM-5.2。


In [ ]:
from openai_codex import ApprovalMode, Codex, Sandbox
from notebook05_story import CORRECTION, STATE_FIELDS, EXACT_FIELDS, input_text, sample_values
from notebook_support import command_receipts, read_json_answer
lab, work, logfile = story.lab, story.work, story.logfile
codex: Codex = lab.codex if lab else None
thread = codex.thread_start(cwd=str(work),
    developer_instructions="仅做只读调查，即使发现异常也不得重启服务。",
    sandbox=Sandbox.full_access, approval_mode=ApprovalMode.deny_all) if codex else None
first = None
if lab:
    turn = thread.turn("完整读取 capacity.log，为 CASE-527 拟定下一步检查计划，暂不作最终归因。概括值得注意的现象，不超过 150 字。")
    with lab.watch(turn):
        first = turn.run()
    lab.record_result(turn, first, show_result=False)


In [ ]:
corrected = None
if lab:
    turn = thread.turn(CORRECTION)
    with lab.watch(turn):
        corrected = turn.run()
    lab.record_result(turn, corrected, show_result=False)
before_request = lab.requests[-1] if lab else {}
before_chars = len(input_text(before_request)) if lab else None
before_history = thread.read(include_turns=True).thread if lab else None
before_turn_ids = {turn.id for turn in before_history.turns} if before_history else set()
story.show_before(first, corrected, CORRECTION, before_request, before_history)


## 上下文更新后，什么不该随之改变？

这里起作用的是压缩后工作集的替换：在原 Thread 发起原生压缩，等新增的 `contextCompaction` 完成项后才继续。
跑完对照“比较项”“压缩前”和“压缩后”，检查待验证是否误变成已排除，以及先查连接池的理由与不许重启是否保留。
结构化复述与真实请求分别核对；输入字符数减少只说明长度变化。


In [ ]:
import time
compaction_turn = None
if lab:
    thread.compact()
    deadline = time.monotonic() + 240
    while time.monotonic() < deadline:
        history = thread.read(include_turns=True).thread
        candidates = [t for t in history.turns if t.id not in before_turn_ids
                      and any(i.root.type == "contextCompaction" for i in t.items)]
        if candidates:
            compaction_turn = candidates[-1]
            if compaction_turn.status.value in ("failed", "interrupted"):
                raise RuntimeError("原生压缩未完成：" + compaction_turn.status.value)
            if compaction_turn.status.value == "completed":
                story._save("compaction.json", {"thread_id": thread.id,
                    "turn": compaction_turn.model_dump(mode="json", by_alias=True)})
                break
        time.sleep(1)
    else:
        raise TimeoutError("240 秒内没有取得新增 contextCompaction 完成项，停止后续模型调用。")
recap_start = len(lab.requests) if lab else 0


In [ ]:
recap = None
if lab:
    turn = thread.turn("先不要调用工具。依据当前保留的历史，只输出一个 JSON 对象，字段 task、release_status、reason、next_step、next_reason、constraint。"
                "task 是任务编号，release_status 只能填已确认、已排除或待验证；其他字段自然说明保留的判断依据、下一步、选择下一步的理由和操作限制。不确定的地方明确说明。")
    with lab.watch(turn):
        recap = turn.run()
    lab.record_result(turn, recap, show_result=False)
state = read_json_answer(recap.final_response, STATE_FIELDS) if recap else {}
after_request = lab.requests[recap_start] if lab else {}
after_chars = len(input_text(after_request)) if lab else None
history_after = thread.read(include_turns=True).thread if lab else None
story.show_after(compaction_turn, recap, state, before_chars, after_chars, history_after, recap_start)


## Codex 怎样做 Compaction？

这里起作用的是完整历史与当前输入的分离：追问两条采样的精确数值，观察模型直接回答还是回查磁盘原文。
跑完对照“保存位置”和“采样原文是否可找到”，同时核对原调查轮次是否仍能从完整历史读到。
当前工作集、完整 Thread 历史与 capacity.log 分开查看；litellm 记录的请求用于判断回答时是否已有依据。


In [ ]:
reload_start = len(lab.requests) if lab else 0
previous_calls = {r["call_id"] for r in command_receipts(lab.requests)} if lab else set()
recovered = None
if lab:
    turn = thread.turn("请给出 sample=0073 与 sample=0127 的 elapsed_us 数值及二者差值。若现有依据不足，不要猜，请自行补足后再回答。"
                    "只输出 JSON 对象，字段 sample_0073、sample_0127、difference，值都为整数，difference 是后者减前者。")
    with lab.watch(turn):
        recovered = turn.run()
    lab.record_result(turn, recovered, show_result=False)
recall_requests = lab.requests[reload_start:] if lab else []
new_reads = [r for r in command_receipts(recall_requests) if r["call_id"] not in previous_calls]
exact = read_json_answer(recovered.final_response, EXACT_FIELDS) if recovered else None


In [ ]:
disk_text = logfile.read_text(encoding="utf-8")
expected = sample_values(disk_text)
expected["difference"] = expected["sample_0127"] - expected["sample_0073"]
after = story.snapshot(work)
print("输入文件：", "未改变" if after == story.input_snapshot else "有变化")
story.show_storage(before_request, after_request, history_after, recovered, exact, expected, new_reads, recall_requests, after)
